In [2]:
"""
train.py — Signature Verification  (ArcFace + Contrastive, v8)
==============================================================
Speed fix: pre-augmented tensor cache for training.

Problem in v7:
  train_transform (PIL, CPU) runs in __getitem__ with num_workers=0
  → single CPU thread feeds GPU → GPU spikes then starves → 3-5 min/epoch

Solution:
  At startup, apply train_transform N=10 times per unique train image.
  Stores N tensors per path. __getitem__ picks one randomly.
  → zero per-batch CPU cost → GPU never waits → back to ~1-2 min/epoch

Memory estimate (your dataset):
  ~2500 unique train images × 10 versions × 64×64×1 × 4 bytes ≈ 390 MB
  Well within 16 GB RAM budget.

Augmentation quality:
  train_transform is applied N times with DIFFERENT random seeds each time
  → Pad→RandomAffine→Perspective→Blur→Resize order fully preserved
  → Each epoch __getitem__ picks a different version → effective diversity

Val/Test: tensor cache (test_transform baked in once) — unchanged from v7.
num_workers=0: tensor dicts cannot be pickled, and with data in RAM it's fast.
"""

import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm

from model import model_final
from dataset import (
    train_pairs,  train_labels,
    val_pairs,    val_labels,
    test_pairs,   test_labels,
    image_cache,
    genuine_by_author,
    train_transform,
    test_transform,
)


# ─────────────────────────────────────────────────────────────────────────────
# DEVICE + SEEDS
# ─────────────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name()}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True


# ─────────────────────────────────────────────────────────────────────────────
# UNWRAP torch.compile
# ─────────────────────────────────────────────────────────────────────────────
def unwrap(m):
    if hasattr(m, '_orig_mod'): return m._orig_mod
    if hasattr(m, 'module'):    return m.module
    return m

raw_model = unwrap(model_final).to(device)


# ─────────────────────────────────────────────────────────────────────────────
# PRE-AUGMENTED TENSOR CACHE  (training)
#
# For each unique training image path, apply train_transform N times,
# each time with a different random state → N distinct augmented tensors.
# __getitem__ picks one at random → zero per-batch CPU cost.
#
# train_transform order is fully preserved:
#   Resize → Grayscale → Pad(8) → RandomAffine → Perspective → Blur
#   → Resize → Invert → ToTensor → Normalize
# ─────────────────────────────────────────────────────────────────────────────
def build_augmented_cache(pairs, transform, n_versions=10, desc="Augmenting"):
    """
    Returns dict: {path: [tensor_v0, tensor_v1, ..., tensor_vN-1]}
    Each tensor is a different random augmentation of the same PIL image.
    Built from image_cache (PIL already in RAM) — zero disk I/O.
    """
    unique = list({p for pair in pairs for p in pair})
    cache  = {}
    for p in tqdm(unique, desc=f"  {desc}", leave=False):
        pil = image_cache[p]
        cache[p] = [transform(pil.copy()) for _ in range(n_versions)]
    return cache


def build_tensor_cache(pairs, transform, desc="Caching"):
    """Single-version cache for val/test (deterministic transform)."""
    unique = list({p for pair in pairs for p in pair})
    cache  = {}
    for p in tqdm(unique, desc=f"  {desc}", leave=False):
        cache[p] = transform(image_cache[p].copy())
    return cache


# ─────────────────────────────────────────────────────────────────────────────
# Number of pre-augmented versions per image.
# 10 = good diversity, ~390 MB for your dataset.
# Reduce to 5 if RAM is tight, increase to 20 for more diversity.
# ─────────────────────────────────────────────────────────────────────────────
N_AUG = 10

print(f"\nBuilding pre-augmented train cache  "
      f"(train_transform × {N_AUG} versions per image) ...")
train_aug_cache = build_augmented_cache(
    train_pairs, train_transform, n_versions=N_AUG, desc="Train augment")

unique_train = len(train_aug_cache)
mb_train     = (unique_train * N_AUG *
                train_aug_cache[next(iter(train_aug_cache))][0].nelement() *
                train_aug_cache[next(iter(train_aug_cache))][0].element_size()) / 1e6
print(f"  Train aug cache : {unique_train} images × {N_AUG} = "
      f"{unique_train * N_AUG:,} tensors  ({mb_train:.0f} MB)")

print("Building val/test tensor caches  (test_transform, 1 version) ...")
val_tensor_cache  = build_tensor_cache(val_pairs,  test_transform, desc="Val")
test_tensor_cache = build_tensor_cache(test_pairs, test_transform, desc="Test")
mb = lambda c: sum(t.nelement()*t.element_size() for t in c.values()) / 1e6
print(f"  Val  cache : {len(val_tensor_cache)}  images  ({mb(val_tensor_cache):.0f} MB)")
print(f"  Test cache : {len(test_tensor_cache)} images  ({mb(test_tensor_cache):.0f} MB)")


# ─────────────────────────────────────────────────────────────────────────────
# DATASETS
# ─────────────────────────────────────────────────────────────────────────────
def _author_from_path(path):
    try:    return os.path.basename(path).split('_')[1]
    except: return "unknown"


class TrainPairDataset(Dataset):
    """
    __getitem__: pick a random pre-augmented version for each image.
    Cost = 2 list lookups + randint → effectively zero CPU time per batch.
    Augmentation diversity = N_AUG different versions of every image.
    Returns: (img1, img2, label, author1_id, author2_id)
    """
    def __init__(self, pairs, labels, aug_cache, author2id, n_versions):
        self.pairs      = pairs
        self.labels     = labels
        self.aug_cache  = aug_cache
        self.author2id  = author2id
        self.n          = n_versions

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        # Pick independent random versions for each image
        img1 = self.aug_cache[p1][random.randrange(self.n)]
        img2 = self.aug_cache[p2][random.randrange(self.n)]
        a1   = self.author2id.get(_author_from_path(p1), 0)
        a2   = self.author2id.get(_author_from_path(p2), 0)
        return (
            img1,
            img2,
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(a1,               dtype=torch.long),
            torch.tensor(a2,               dtype=torch.long),
        )


class EvalPairDataset(Dataset):
    """
    __getitem__: pure tensor dict lookup — zero transform cost.
    Returns: (img1, img2, label, author1_id, author2_id)
    """
    def __init__(self, pairs, labels, tensor_cache, author2id):
        self.pairs        = pairs
        self.labels       = labels
        self.tensor_cache = tensor_cache
        self.author2id    = author2id

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        a1     = self.author2id.get(_author_from_path(p1), 0)
        a2     = self.author2id.get(_author_from_path(p2), 0)
        return (
            self.tensor_cache[p1],
            self.tensor_cache[p2],
            torch.tensor(self.labels[idx], dtype=torch.float32),
            torch.tensor(a1,               dtype=torch.long),
            torch.tensor(a2,               dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# LOSSES
# ─────────────────────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, num_classes, s=32.0, m=0.50):
        super().__init__()
        self.s = s; self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        W      = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(embeddings.float(), W.float())
        sine   = torch.sqrt((1.0 - cosine.pow(2)).clamp(1e-9, 1.0))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        oh     = torch.zeros_like(cosine)
        oh.scatter_(1, labels.view(-1, 1), 1.0)
        logits = (oh * phi + (1.0 - oh) * cosine) * self.s
        return F.cross_entropy(logits, labels)


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        d   = F.pairwise_distance(emb1.float(), emb2.float(), p=2)
        pos = labels       * d.pow(2)
        neg = (1 - labels) * F.relu(self.margin - d).pow(2)
        return (pos + neg).mean()


class SignatureVerificationLoss(nn.Module):
    def __init__(self, num_classes, emb_dim=256, s=32.0, m=0.50,
                 margin=1.0, lambda_arc=0.6, lambda_con=0.4):
        super().__init__()
        self.arcface     = ArcFaceLoss(emb_dim, num_classes, s=s, m=m)
        self.contrastive = ContrastiveLoss(margin=margin)
        self.la = lambda_arc
        self.lc = lambda_con

    def forward(self, emb1, emb2, pair_labels, auth1, auth2):
        arc   = (self.arcface(emb1, auth1) + self.arcface(emb2, auth2)) / 2.0
        con   = self.contrastive(emb1, emb2, pair_labels)
        total = self.la * arc + self.lc * con
        return total, arc.item(), con.item()


# ─────────────────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_distances(model, loader):
    model.eval()
    dists, labs = [], []
    for img1, img2, labels, _, _ in loader:
        with autocast('cuda', enabled=torch.cuda.is_available()):
            e1, e2 = model(img1.to(device, non_blocking=True),
                           img2.to(device, non_blocking=True))
        dists.append(F.pairwise_distance(e1.float(), e2.float()).cpu().numpy())
        labs.append(labels.numpy())
    return np.concatenate(dists), np.concatenate(labs)


def best_threshold(distances, labels):
    best_acc, best_t = 0.0, 0.5
    for t in np.linspace(distances.min(), distances.max(), 200):
        acc = accuracy_score(labels, (distances < t).astype(int))
        if acc > best_acc: best_acc, best_t = acc, t
    return best_t, best_acc


def evaluate(model, loader, split="Val"):
    dists, labels = compute_distances(model, loader)
    t, acc = best_threshold(dists, labels)
    try:    auc = roc_auc_score(labels, -dists)
    except: auc = 0.5
    preds   = (dists < t).astype(int)
    genuine = labels == 1; forged = labels == 0
    far     = float(np.mean(preds[forged]  == 1)) if forged.any()  else 0.0
    frr     = float(np.mean(preds[genuine] == 0)) if genuine.any() else 0.0
    gen_m   = float(dists[genuine].mean()) if genuine.any() else 0.0
    forg_m  = float(dists[forged].mean())  if forged.any()  else 0.0
    print(f"  [{split}] Acc={acc:.4f}  AUC={auc:.4f}  "
          f"FAR={far:.4f}  FRR={frr:.4f}  "
          f"Gap={forg_m-gen_m:.3f}  Thr={t:.4f}")
    return {"acc": acc, "auc": auc, "far": far, "frr": frr,
            "threshold": t, "gap": forg_m - gen_m}


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN EPOCH
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model, criterion, optimizer, scheduler, scaler, loader, epoch):
    model.train()
    tot = tot_arc = tot_con = 0.0
    use_amp = torch.cuda.is_available()
    pbar    = tqdm(loader, desc=f"Epoch {epoch:03d} [Train]", leave=False)

    for img1, img2, pair_labels, auth1, auth2 in pbar:
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=use_amp):
            e1, e2         = model(img1, img2)
            loss, arc, con = criterion(e1, e2, pair_labels, auth1, auth2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        tot += loss.item(); tot_arc += arc; tot_con += con
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         arc=f"{arc:.4f}", con=f"{con:.4f}")

    n = len(loader)
    print(f"  [Train] Loss={tot/n:.4f}  "
          f"ArcFace={tot_arc/n:.4f}  Contrastive={tot_con/n:.4f}")
    return tot / n


# ─────────────────────────────────────────────────────────────────────────────
# VAL EPOCH
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def val_epoch(model, criterion, loader):
    model.eval()
    tot = 0.0
    use_amp = torch.cuda.is_available()
    for img1, img2, pair_labels, auth1, auth2 in loader:
        img1        = img1.to(device, non_blocking=True)
        img2        = img2.to(device, non_blocking=True)
        pair_labels = pair_labels.to(device, non_blocking=True)
        auth1       = auth1.to(device, non_blocking=True)
        auth2       = auth2.to(device, non_blocking=True)
        with autocast('cuda', enabled=use_amp):
            e1, e2     = model(img1, img2)
            loss, _, _ = criterion(e1, e2, pair_labels, auth1, auth2)
        tot += loss.item()
    print(f"  [Val  ] Loss={tot/len(loader):.4f}")
    return tot / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def train(
    model,
    num_epochs         = 40,
    lr                 = 3e-4,
    weight_decay       = 1e-4,
    arcface_margin     = 0.50,
    arcface_scale      = 32.0,
    contrastive_margin = 1.0,
    lambda_arc         = 0.6,
    lambda_con         = 0.4,
    save_dir           = "./checkpoints",
    patience           = 10,
):
    os.makedirs(save_dir, exist_ok=True)

    author2id   = {a: i for i, a in enumerate(sorted(genuine_by_author.keys()))}
    num_classes = len(author2id)
    print(f"\nAuthors (ArcFace classes) : {num_classes}")

    # ── Datasets ─────────────────────────────────────────────────────────────
    train_ds = TrainPairDataset(
        train_pairs, train_labels, train_aug_cache, author2id, N_AUG)
    val_ds   = EvalPairDataset(
        val_pairs,   val_labels,   val_tensor_cache,  author2id)
    test_ds  = EvalPairDataset(
        test_pairs,  test_labels,  test_tensor_cache, author2id)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                              num_workers=0, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                              num_workers=0, pin_memory=pin)

    print(f"Loaders (num_workers=0, all data in RAM):")
    print(f"  train : {len(train_loader)} batches  ({len(train_ds):,} pairs)")
    print(f"  val   : {len(val_loader)}  batches  ({len(val_ds):,} pairs)")
    print(f"  test  : {len(test_loader)} batches  ({len(test_ds):,} pairs)")

    # ── Loss / Optimizer / Scheduler ─────────────────────────────────────────
    criterion = SignatureVerificationLoss(
        num_classes, emb_dim=256, s=arcface_scale, m=arcface_margin,
        margin=contrastive_margin, lambda_arc=lambda_arc, lambda_con=lambda_con,
    ).to(device)

    optimizer = AdamW([
        {"params": model.parameters(),
         "lr": lr, "weight_decay": weight_decay},
        {"params": criterion.arcface.parameters(),
         "lr": lr * 0.1, "weight_decay": weight_decay},
    ])

    scheduler = OneCycleLR(
        optimizer,
        max_lr=[lr, lr * 0.1],
        total_steps=num_epochs * len(train_loader),
        pct_start=0.1, anneal_strategy="cos",
        div_factor=25, final_div_factor=1e4,
    )

    scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

    # ── Training loop ─────────────────────────────────────────────────────────
    best_auc = 0.0; best_epoch = 0; no_improve = 0
    history  = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}

    print("\n" + "="*65)
    print("TRAINING  —  ArcFace + Contrastive")
    print(f"  Train  : pre-aug tensor cache ({N_AUG} versions/img), zero CPU per batch")
    print(f"  Val    : tensor cache, zero transform cost")
    print(f"  Epochs : {num_epochs}   LR : {lr}   Patience : {patience}")
    print(f"  λ_arc={lambda_arc}  λ_con={lambda_con}  margin={contrastive_margin}")
    print("="*65)

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        print(f"\nEpoch {epoch}/{num_epochs}")
        tl = train_epoch(model, criterion, optimizer, scheduler,
                         scaler, train_loader, epoch)
        vl = val_epoch(model, criterion, val_loader)
        vm = evaluate(model, val_loader, "Val")
        print(f"  Time: {time.time()-t0:.0f}s")

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_auc"].append(vm["auc"])
        history["val_acc"].append(vm["acc"])

        if vm["auc"] > best_auc:
            best_auc, best_epoch, no_improve = vm["auc"], epoch, 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "arc_state": criterion.arcface.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "val_auc": best_auc, "val_acc": vm["acc"],
                "threshold": vm["threshold"], "author2id": author2id,
            }, os.path.join(save_dir, "best_model.pt"))
            print(f"  ✓ Best saved  AUC={best_auc:.4f}  Acc={vm['acc']:.4f}")
        else:
            no_improve += 1
            print(f"  No improve {no_improve}/{patience}")
            if no_improve >= patience:
                print(f"  Early stop at epoch {epoch}"); break

    # ── Final test ────────────────────────────────────────────────────────────
    print("\n" + "="*65 + "\nFINAL TEST EVALUATION\n" + "="*65)
    ckpt = torch.load(os.path.join(save_dir, "best_model.pt"), map_location=device,weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    tm = evaluate(model, test_loader, "Test")

    print(f"\nBest epoch={best_epoch}  Val AUC={best_auc:.4f}")
    print(f"Test AUC={tm['auc']:.4f}  Acc={tm['acc']:.4f}  "
          f"FAR={tm['far']:.4f}  FRR={tm['frr']:.4f}  Gap={tm['gap']:.3f}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    dists, labels_np = compute_distances(model, test_loader)
    gen_d  = dists[labels_np == 1]
    forg_d = dists[labels_np == 0]

    try:
        import seaborn as sns
        plt.figure(figsize=(9, 5))
        sns.histplot(gen_d,  color='royalblue', label='Genuine',
                     kde=True, stat='density', alpha=0.6)
        sns.histplot(forg_d, color='tomato',    label='Forged',
                     kde=True, stat='density', alpha=0.6)
    except ImportError:
        plt.figure(figsize=(9, 5))
        plt.hist(gen_d,  bins=50, color='royalblue',
                 label='Genuine', density=True, alpha=0.6)
        plt.hist(forg_d, bins=50, color='tomato',
                 label='Forged',  density=True, alpha=0.6)

    plt.axvline(tm["threshold"], color='green', linestyle='--',
                label=f"Thr ({tm['threshold']:.3f})")
    plt.xlabel('L2 Distance'); plt.ylabel('Density')
    plt.title('Test Distance Distribution')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "distance_distribution.png"), dpi=150)
    plt.close()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].plot(history["val_auc"])
    axes[1].set_title("Val AUC"); axes[1].set_xlabel("Epoch")
    axes[2].plot(history["val_acc"])
    axes[2].set_title("Val Accuracy"); axes[2].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150)
    plt.close()

    print(f"Plots → {save_dir}/")
    return model, history, tm


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    trained_model, history, test_metrics = train(
        model              = raw_model,
        num_epochs         = 40,
        lr                 = 3e-4,
        weight_decay       = 1e-4,
        arcface_margin     = 0.50,
        arcface_scale      = 32.0,
        contrastive_margin = 1.2,
        lambda_arc         = 0.6,
        lambda_con         = 0.4,
        save_dir           = "./checkpoints",
        patience           = 10,
    )

Device : cuda
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU

Building pre-augmented train cache  (train_transform × 10 versions per image) ...


  Train aug cache : 5675 images × 10 = 56,750 tensors  (930 MB)
Building val/test tensor caches  (test_transform, 1 version) ...


  Val  cache : 789  images  (13 MB)
  Test cache : 859 images  (14 MB)

Authors (ArcFace classes) : 268
Loaders (num_workers=0, all data in RAM):
  train : 118 batches  (60,240 pairs)
  val   : 19  batches  (9,270 pairs)
  test  : 19 batches  (9,450 pairs)

TRAINING  —  ArcFace + Contrastive
  Train  : pre-aug tensor cache (10 versions/img), zero CPU per batch
  Val    : tensor cache, zero transform cost
  Epochs : 40   LR : 0.0003   Patience : 10
  λ_arc=0.6  λ_con=0.4  margin=1.2

Epoch 1/40


Epoch 001 [Train]:   0%|                                                                       | 0/118 [00:00<?, ?it/s]C:\Users\gupta\AppData\Local\Temp\ipykernel_4796\508805282.py:330: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
                                                                                                                       

  [Train] Loss=1.7138  ArcFace=2.6236  Contrastive=0.3491
  [Val  ] Loss=17.6219
  [Val] Acc=0.8531  AUC=0.9347  FAR=0.1836  FRR=0.1102  Gap=0.471  Thr=0.9161
  Time: 40s
  ✓ Best saved  AUC=0.9347  Acc=0.8531

Epoch 2/40


...

  [Train] Loss=0.2512  ArcFace=0.2187  Contrastive=0.3000
  [Val  ] Loss=19.0199
  [Val] Acc=0.8360  AUC=0.9135  FAR=0.2356  FRR=0.0923  Gap=0.443  Thr=1.0005
  Time: 39s
  No improve 6/10

Epoch 9/40


  [Train] Loss=0.2213  ArcFace=0.1702  Contrastive=0.2978
  [Val  ] Loss=18.8048
  [Val] Acc=0.8334  AUC=0.9155  FAR=0.2041  FRR=0.1290  Gap=0.430  Thr=0.9819
  Time: 39s
  No improve 7/10

Epoch 10/40


  [Train] Loss=0.2025  ArcFace=0.1413  Contrastive=0.2943
  [Val  ] Loss=18.4291
  [Val] Acc=0.8387  AUC=0.9177  FAR=0.1810  FRR=0.1415  Gap=0.429  Thr=0.9869
  Time: 39s
  No improve 8/10

Epoch 11/40


  [Train] Loss=0.1879  ArcFace=0.1190  Contrastive=0.2914
  [Val  ] Loss=18.8272
  [Val] Acc=0.8388  AUC=0.9200  FAR=0.2201  FRR=0.1023  Gap=0.437  Thr=0.9861
  Time: 39s
  No improve 9/10

Epoch 12/40


  [Train] Loss=0.1816  ArcFace=0.1103  Contrastive=0.2886
  [Val  ] Loss=19.0749
  [Val] Acc=0.8292  AUC=0.9173  FAR=0.2412  FRR=0.1003  Gap=0.430  Thr=1.0239
  Time: 39s
  No improve 10/10
  Early stop at epoch 12

FINAL TEST EVALUATION
  [Test] Acc=0.8255  AUC=0.9100  FAR=0.2665  FRR=0.0825  Gap=0.455  Thr=1.0566

Best epoch=2  Val AUC=0.9355
Test AUC=0.9100  Acc=0.8255  FAR=0.2665  FRR=0.0825  Gap=0.455
Plots → ./checkpoints/
